# Modem dan analyzer lab: perintah AT dan ASTM

Dua veteran port serial yang masih ada di mana-mana: modul seluler yang dikendalikan dengan perintah AT, dan analyzer laboratorium yang mengirim hasil dengan ASTM E1394. Keduanya berjalan terhadap simulator di sini.

## Persiapan
> Bekerja dari hasil clone? Jalankan `dotnet pack -c Release -o artifacts/packages` di root repo lalu tambahkan
> `#i "nuget: <repo>/artifacts/packages"` sebelum baris `#r`.

In [ ]:
#r "nuget: IoTCom.Net, 0.10.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.10.0-preview.1"

## Modul seluler
Perintah diakhiri OK atau error; unsolicited result code (URC) seperti `+CEREG` dan `+CMTI` bisa datang kapan saja.

In [ ]:
using IoTCom.Net;
using IoTCom.Net.Protocols.AtCommand;
using IoTCom.Net.Transports;

var modemLink = new InMemoryTransportListener("modem");
await using var module = AtModemSimulator.Create(o => { o.ListenInMemory(modemLink); o.RegistrationDelay = TimeSpan.FromMilliseconds(300); });
await module.StartAsync();
await using var modem = AtModem.Create(o => o.UseInMemory(modemLink));
await modem.ConnectAsync();
await modem.SendCheckedAsync("AT+CEREG=2");
await Task.Delay(600);
var info = await modem.GetInfoAsync();
Console.WriteLine($"{info.Model} · {info.Registration} on {info.Operator} ({info.AccessTechnology}) · {info.Signal}");
Console.WriteLine(await modem.SendAsync("AT+CPIN=\"0000\""));

## SMS masuk dan keluar

In [ ]:
await modem.SendSmsAsync("+6281234567890", "Pompa air nyala");
var inbox = new TaskCompletionSource<AtUrc>();
modem.UrcReceived += (_, u) => { if (u.Name == "+CMTI") inbox.TrySetResult(u); };
module.DeliverSms("+6289876543210", "STATUS?");
var cmti = await inbox.Task;
var sms = await modem.ReadSmsAsync(int.Parse(cmti.Values[1]));
Console.WriteLine($"{sms.Sender}: {sms.Text}");

## Analyzer kimia lewat ASTM
Analyzer mengirim ENQ, lalu satu record berbingkai setiap kali, masing-masing di-ACK; frame yang rusak dijawab NAK dan dikirim ulang.

In [ ]:
using IoTCom.Net.Protocols.Astm;

var labLink = new InMemoryTransportListener("lab");
await using var lis = AstmReceiver.Create(o => o.ListenInMemory(labLink));
await lis.StartAsync();
var resultArrived = new TaskCompletionSource<AstmMessage>();
lis.MessageReceived += (_, m) => resultArrived.TrySetResult(m);
await using var analyzer = AstmSender.Create(o => o.UseInMemory(labLink));
await analyzer.ConnectAsync();
analyzer.CorruptNextFrame = true;
await analyzer.SendAsync(new AnalyzerSimulator().NextResult());
var labResult = await resultArrived.Task;
foreach (var r in labResult.Results) Console.WriteLine($"{r.TestCode,-5} {r.Value,7} {r.Units,-7} {r.Flag}");
Console.WriteLine($"retransmitted frames: {analyzer.Retransmissions}");

## Jembatan ke HL7
Kebanyakan rumah sakit menginginkan hasil sebagai HL7 ORU^R01.

In [ ]:
Console.WriteLine(AstmToHl7.ToOru(labResult).Encode().Replace("\r", "\n"));

## Lebih lanjut
`iotcom at info --sim`, `iotcom astm listen --hl7`, dan sampel AstmAnalyzerBridge. Data sintetis saja; bukan perangkat medis. Lihat `docs/id/protocols/at-commands.md` dan `docs/id/protocols/astm.md`.

*IoTCom.Net — dibuat oleh Gravicode Studios dipimpin oleh Kang Fadhil.*